# Stage 2 (Exoplanet): Trajectory Collection (Colab A100)

Runs `src/exoplanet_data_construction.py` to build the dataset (NASA TAP queries, ~30s),
then `src/exoplanet_agent_loop.py` on A100 with bf16 precision.

Mirrors `stage2_colab.ipynb`. Uses the same Llama-3.1-8B-Instruct revision pinned in `config.yaml`.

**Setup:** Run cells in order. Smoke test first (5 trajectories, ~2 min), then the full 2000-trajectory run after manual review of the smoke output.

In [ ]:
# Cell 1: Check GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name()}")
    print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")

In [ ]:
# Cell 2: Clone repo (use the tanner/exoplanet_data branch until it lands on main)
!git clone --branch tanner/exoplanet_data https://github.com/agastyasridharan/bayes-and-confused.git /content/bayes-and-confused

import os
os.chdir('/content/bayes-and-confused')
!ls

In [ ]:
# Cell 3: Install dependencies
!pip install -q pyyaml transformers accelerate astroquery

In [ ]:
# Cell 4: Authenticate with Hugging Face (needed for Llama gated model)
from huggingface_hub import login
login()  # paste your HF token when prompted

In [ ]:
# Cell 5: Build the dataset (200 real planets + 200 next-letter perturbations)
# Hits NASA Exoplanet Archive TAP service (no auth, ~30s).
!python src/exoplanet_data_construction.py

In [ ]:
# Cell 6: Run trajectory collection
# Smoke test first (5 trajectories, ~2 min) — review the printed prompts and
# extraction tokens before uncommenting the full run below.
!python src/exoplanet_agent_loop.py --smoke-test
# Uncomment below for the full 2000-trajectory run after verifying smoke output:
# !python src/exoplanet_agent_loop.py

In [ ]:
# Cell 7: Quick inspection of results (gate-2 review: read 30 random empty-side)
import json, random
random.seed(42)

with open('data/trajectories/exoplanet_trajectories.json') as f:
    trajs = json.load(f)

empty = [t for t in trajs if t['side'] == 'empty']
present = [t for t in trajs if t['side'] == 'data_present']
print(f'Total: {len(trajs)} ({len(empty)} empty, {len(present)} data-present)')

# Show 30 random empty-side trajectories for gate review
print('\n=== 30 RANDOM EMPTY-SIDE TRAJECTORIES ===')
for t in random.sample(empty, 30):
    print(f"\n--- [{t['trajectory_id']}] {t['system_prompt_variant']} | {t['pl_name']} | {t['property']} ---")
    print(f"Query: {t['user_query']}")
    print(f"Response: {t['assistant_response'][:500]}")
    print()

In [ ]:
# Cell 8: Download results (trajectories + activations + dataset)
from google.colab import files
!zip -qr exoplanet_trajectories.zip data/trajectories/ data/exoplanets.json
files.download('exoplanet_trajectories.zip')